# Analyzing Students' Mental Health with SQL

This notebook explores whether studying in a foreign country is associated with differences in mental health among university students. The data comes from a Japanese international university that surveyed its students in 2018 and published the study in 2019.

Previous research suggests that:
- **International students** have a higher risk of mental health difficulties than the general population.
- **Social connectedness** (feeling that you belong to a social group) and **acculturative stress** (stress related to adapting to a new culture) are important predictors of depression.

In this project, I use **PostgreSQL** to:
1. Inspect the raw `students` table.
2. Filter the data to international students.
3. Aggregate mental health scores by length of stay in Japan.
4. Summarize my findings in plain language.

## Data description

The main table used in this project is called `students`.  
Below are the key fields used in the analysis:

| Field Name   | Description                                           |
|-------------|--------------------------------------------------------|
| `inter_dom` | Type of student (international or domestic)           |
| `stay`      | Current length of stay in Japan (years)               |
| `todep`     | Total depression score (PHQ-9 test)                   |
| `tosc`      | Total social connectedness score (SCS test)           |
| `toas`      | Total acculturative stress score (ASISS test)        |

The analysis focuses on **international students** (`inter_dom = 'Inter'`).

In [ ]:
-- Preview the raw students table
SELECT *
FROM students
LIMIT 10;

## Filtering to international students

The original dataset includes both domestic and international students.  
Since the research question focuses on international students, the first step is to filter the data to rows where `inter_dom = 'Inter'`.

In [ ]:
-- Filter to international students only
SELECT *
FROM students
WHERE inter_dom = 'Inter'
LIMIT 10;

## Aggregating mental health scores by length of stay

To understand how the length of stay might relate to mental health, I group international students by `stay` (years in Japan) and calculate:

- `count_int`: number of international students in each group.
- `average_phq`: average PHQ-9 depression score (`todep`), rounded to 2 decimals.
- `average_scs`: average social connectedness score (`tosc`), rounded to 2 decimals.
- `average_as`: average acculturative stress score (`toas`), rounded to 2 decimals.

I keep only groups with at least one student and sort the results by `stay` in descending order.

In [ ]:
-- Aggregate mental health scores by length of stay for international students
SELECT
    stay,
    COUNT(*) AS count_int,
    ROUND(AVG(todep), 2) AS average_phq,
    ROUND(AVG(tosc), 2) AS average_scs,
    ROUND(AVG(toas), 2) AS average_as
FROM students
WHERE inter_dom = 'Inter'
GROUP BY stay
HAVING COUNT(*) > 0
ORDER BY stay DESC;

## Interpreting the aggregated results

The query above returns a table with:

- **One row per length of stay (`stay`)** among international students.
- **Five columns**: `stay`, `count_int`, `average_phq`, `average_scs`, and `average_as`.

Some key observations from the output (values will depend on the exact data):

- The **largest groups** of international students tend to have shorter stays (e.g., 1–3 years), while very long stays have only a few students.
- The **average PHQ-9 scores** are relatively close across different lengths of stay, without a strong, monotonic increase or decrease.
- **Social connectedness (SCS)** and **acculturative stress (ASISS)** also vary somewhat by length of stay, but the patterns are not perfectly linear.
- For stay values with only **one or very few students**, the averages are less reliable and should be interpreted with caution.

## Conclusions

Based on this simple aggregation:

- There is **no clear, strong evidence** in this summary table that a longer stay in Japan is consistently associated with either much higher or much lower depression scores among international students.
- Differences in average PHQ-9 scores between stay groups are **relatively small**, and some groups have very few students, which makes their averages unstable.
- A more robust analysis would require:
  - Formal statistical tests (e.g., regression models, significance tests),
  - Controlling for other variables (age, academic level, language proficiency, etc.),
  - And inspecting the full distribution of scores, not just the means.

In other words, this SQL analysis is a **useful descriptive starting point**, but it is not enough on its own to make strong causal claims such as "a longer and more isolated stay directly causes higher depression levels".